<a href="https://colab.research.google.com/github/saim9211/Machine-Learning-Stuff/blob/main/Decision_tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import kagglehub
path = kagglehub.dataset_download("uciml/red-wine-quality-cortez-et-al-2009")

Using Colab cache for faster access to the 'red-wine-quality-cortez-et-al-2009' dataset.


In [36]:
import pandas as pd

df = pd.read_csv(f'{path}/winequality-red.csv')
display(df.sample(3))

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
696,7.0,0.650,0.02,2.10,0.066,8.0,25.0,0.99720,3.47,0.67,9.5,6
914,7.3,0.305,0.39,1.20,0.059,7.0,11.0,0.99331,3.29,0.52,11.5,6
357,10.5,0.420,0.66,2.95,0.116,12.0,29.0,0.99700,3.24,0.75,11.7,7


In [37]:
df['quality']

,quality
0,5
1,5
2,5
3,6
4,5
...,...
1594,5
1595,6
1596,6
1597,5


In [38]:
import numpy as np
import matplotlib as plt
import seaborn as sns

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [40]:
df.isnull().sum()

,0
fixed acidity,0
volatile acidity,0
citric acid,0
residual sugar,0
chlorides,0
free sulfur dioxide,0
total sulfur dioxide,0
density,0
pH,0
sulphates,0


In [41]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


In [42]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

In [43]:
x_test, x_train, y_test, y_train = train_test_split(df.drop('quality', axis=1), df['quality'], test_size=0.2, random_state=42)

In [44]:
x_test.shape, x_train.shape, y_test.shape, y_train.shape

((1279, 11), (320, 11), (1279,), (320,))

In [45]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Define the base pipeline with a scaler and a Decision Tree Classifier
base_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# Define the parameter grid for the DecisionTreeClassifier
param_grid = {
    'classifier__max_depth': [3, 5, 7, None],  # None means no limit
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__criterion': ['gini', 'entropy']
}

# Create a GridSearchCV object that wraps the pipeline
# The 'pipeline' variable will now hold the GridSearchCV object
pipeline = GridSearchCV(estimator=base_pipeline,
                        param_grid=param_grid,
                        cv=5, # 5-fold cross-validation
                        scoring='accuracy', # Metric to optimize
                        n_jobs=-1, # Use all available cores
                        verbose=1) # Display progress

In [46]:
pip=pipeline.fit(x_train, y_train)
pip

Fitting 5 folds for each of 72 candidates, totalling 360 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('classifier',
                                        DecisionTreeClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__criterion': ['gini', 'entropy'],
                         'classifier__max_depth': [3, 5, 7, None],
                         'classifier__min_samples_leaf': [1, 2, 4],
                         'classifier__min_samples_split': [2, 5, 10]},
             scoring='accuracy', verbose=1)

In [47]:
print("Best Parameters found:", pipeline.best_params_)
print("Best Cross-Validation Score:", pipeline.best_score_)

Best Parameters found: {'classifier__criterion': 'gini', 'classifier__max_depth': 3, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2}
Best Cross-Validation Score: 0.521875


In [48]:
# Evaluate the model with the best estimator
y_pred_tuned = pipeline.best_estimator_.predict(x_test)

print("Accuracy Score (tuned model):", accuracy_score(y_test, y_pred_tuned))
print("\nConfusion Matrix (tuned model):\n", confusion_matrix(y_test, y_pred_tuned))
print("\nClassification Report (tuned model):\n", classification_report(y_test, y_pred_tuned))

Accuracy Score (tuned model): 0.5496481626270524

Confusion Matrix (tuned model):
 [[  0   0   6   3   0   0]
 [  0   0  23  19   1   0]
 [  0   0 420 105  26   0]
 [  0   0 201 232  73   0]
 [  0   0  17  89  51   0]
 [  0   0   1   9   3   0]]

Classification Report (tuned model):
               precision    recall  f1-score   support

           3       0.00      0.00      0.00         9
           4       0.00      0.00      0.00        43
           5       0.63      0.76      0.69       551
           6       0.51      0.46      0.48       506
           7       0.33      0.32      0.33       157
           8       0.00      0.00      0.00        13

    accuracy                           0.55      1279
   macro avg       0.24      0.26      0.25      1279
weighted avg       0.51      0.55      0.53      1279



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [49]:
y_pred = pipeline.predict(x_test)

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy Score: 0.5496481626270524

Confusion Matrix:
 [[  0   0   6   3   0   0]
 [  0   0  23  19   1   0]
 [  0   0 420 105  26   0]
 [  0   0 201 232  73   0]
 [  0   0  17  89  51   0]
 [  0   0   1   9   3   0]]

Classification Report:
               precision    recall  f1-score   support

           3       0.00      0.00      0.00         9
           4       0.00      0.00      0.00        43
           5       0.63      0.76      0.69       551
           6       0.51      0.46      0.48       506
           7       0.33      0.32      0.33       157
           8       0.00      0.00      0.00        13

    accuracy                           0.55      1279
   macro avg       0.24      0.26      0.25      1279
weighted avg       0.51      0.55      0.53      1279



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
